# C2: Timeline Analysis

---

## Overview

Analyze time spent at each development stage and identify bottlenecks.

**Metrics:**
- Average time at each stage
- Bottleneck identification
- Comparison by project size
- Seasonal patterns

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Add modules to path
sys.path.insert(0, str(Path.cwd().parent.parent))

from modules.timeline_calculator import (
    calculate_days_between,
    get_stage_durations,
    STATUS_ORDER
)
from modules.data_loader import load_csv

# Configuration
with open('../../config/berkeley_config.json') as f:
    CONFIG = json.load(f)

DATA_DIR = Path(CONFIG['paths']['data_dir'])

print("Pipeline stages for analysis:")
for i, stage in enumerate(STATUS_ORDER, 1):
    print(f"  {i}. {stage}")

## 2. Load Data

In [ ]:
# Load housing projects
housing_path = DATA_DIR / 'housing_projects_FINAL.csv'
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    print(f"\nColumns available: {df.columns.tolist()}")

## 3. Year-Based Analysis

Since we have year data, analyze by filing year.

In [ ]:
# Distribution by year
if df is not None and 'year' in df.columns:
    year_summary = df.groupby('year').agg({
        'address_display': 'count',
        'net_units': ['sum', 'mean']
    }).round(1)
    
    year_summary.columns = ['Projects', 'Total Units', 'Avg Units']
    year_summary = year_summary.sort_index(ascending=False)
    
    print("Projects by Year:")
    display(year_summary)

## 4. Timeline by Project Size

In [ ]:
# Analysis by project size
if df is not None and 'project_size_category' in df.columns:
    size_summary = df.groupby('project_size_category').agg({
        'address_display': 'count',
        'net_units': 'sum',
        'status': lambda x: x.value_counts().index[0]  # Most common status
    }).reset_index()
    
    size_summary.columns = ['Size Category', 'Projects', 'Total Units', 'Most Common Status']
    
    print("Analysis by Project Size:")
    display(size_summary)

## 5. Status Duration Estimates

Estimate typical duration at each status based on year filed.

In [ ]:
# Calculate estimated age of projects
if df is not None and 'year' in df.columns:
    current_year = 2025
    df['years_in_pipeline'] = current_year - df['year']
    
    # Average years by status
    status_age = df.groupby('status').agg({
        'years_in_pipeline': ['mean', 'min', 'max', 'count']
    }).round(1)
    
    status_age.columns = ['Avg Years', 'Min Years', 'Max Years', 'Projects']
    status_age = status_age.sort_values('Avg Years', ascending=False)
    
    print("Time in Pipeline by Current Status:")
    display(status_age)

## 6. Bottleneck Identification

In [ ]:
# Identify bottlenecks (statuses with high project counts and long durations)
if df is not None:
    print("Potential Bottlenecks (high count, long duration):")
    print("="*60)
    
    # Statuses with most projects
    status_counts = df['status'].value_counts()
    
    for status, count in status_counts.head(5).items():
        units = df[df['status'] == status]['net_units'].sum()
        avg_years = df[df['status'] == status]['years_in_pipeline'].mean() if 'years_in_pipeline' in df.columns else 0
        print(f"\n{status}:")
        print(f"  Projects: {count}")
        print(f"  Units: {units:,.0f}")
        print(f"  Avg years in pipeline: {avg_years:.1f}")

## 7. Seasonal Patterns

Note: Requires date-level data for true seasonal analysis.

In [ ]:
# Year-over-year growth
if df is not None and 'year' in df.columns:
    yearly = df.groupby('year')['net_units'].sum().sort_index()
    yearly_growth = yearly.pct_change() * 100
    
    growth_df = pd.DataFrame({
        'Year': yearly.index,
        'Units': yearly.values,
        'YoY Growth %': yearly_growth.values
    }).dropna()
    
    print("Year-over-Year Growth:")
    display(growth_df.round(1))

## 8. Export Analysis

In [ ]:
# Export timeline analysis
if df is not None:
    output_path = DATA_DIR / 'timeline_analysis.csv'
    
    analysis_cols = ['address_display', 'net_units', 'status', 'year']
    if 'years_in_pipeline' in df.columns:
        analysis_cols.append('years_in_pipeline')
    
    df[analysis_cols].to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook analyzed:
- Project distribution by year
- Timeline by project size
- Duration estimates by status
- Potential bottlenecks
- Year-over-year growth

**Next:** Run `C3_proposal_vs_reality.ipynb` to compare proposed vs actual.